# Module 01 — Binary, Hexadecimal, and Bases
### Mathematics of Cryptography · Python Companion

---

Cryptography talks in bits and bytes. Before you can understand XOR, AES, or SHA-256, you need to read the machine's handwriting fluently — binary, hexadecimal, and place value.

This notebook makes those ideas *executable*. You will write Python that converts numbers, splits bytes into nibbles, and watches overflow happen in code.

**What you will build:**
- Functions that convert between binary, decimal, and hex *by hand* (using math, not just built-ins)
- A nibble splitter that mirrors how hex encoding works
- A fixed-width overflow simulator
- A mini hex-decoder for real cryptographic byte sequences

**Prerequisites:** Basic Python (variables, loops, functions). No prior cryptography needed.

---
## Section 1 — Setup and Display Helpers

In [ ]:
# Run this cell first — it defines the display helpers used throughout the notebook.

def show_byte(value, label="value"):
    """Show a value in all three representations, padded for alignment."""
    v = value & 0xFF  # keep it in the byte range 0–255
    print(f"  {label:<16} dec={v:>3}   hex=0x{v:02X}   bin={v:08b}")

def show_conversion(value, from_base, to_base, result, steps=""):
    """Print a labelled conversion with optional working."""
    labels = {2: "binary", 10: "decimal", 16: "hex"}
    print(f"  {labels[from_base]:>8}  →  {labels[to_base]:<8}  :  {value}  =  {result}")
    if steps:
        print(f"  {'':>8}     {'working':8}  :  {steps}")

def separator(title=""):
    """Print a section divider."""
    if title:
        print(f"\n── {title} " + "─" * (50 - len(title)))
    else:
        print("─" * 54)

print("Helpers loaded. Let's start.")

---
## Section 2 — Place Value: the Shared Skeleton

Every positional number system uses the same idea: each digit is multiplied by a *power of the base*.

$$\text{value} = d_m b^m + d_{m-1}b^{m-1} + \cdots + d_1 b^1 + d_0 b^0$$

In Python this formula becomes a loop.

In [ ]:
def place_value_expand(digits, base):
    """
    Show the full place-value expansion of a number given as a list of digits
    (most significant first) and a base.
    Returns the decimal value.
    """
    n = len(digits)
    total = 0
    terms = []
    for i, d in enumerate(digits):
        power = n - 1 - i
        contribution = d * (base ** power)
        total += contribution
        if d != 0:  # only show non-zero terms
            terms.append(f"{d}×{base}^{power}({contribution})")
    print(f"  digits={digits}  base={base}")
    print(f"  expansion: " + " + ".join(terms))
    print(f"  total = {total}")
    return total

separator("Decimal 345 — confirming the formula")
place_value_expand([3, 4, 5], base=10)

separator("Binary 00101101 — same formula, base 2")
place_value_expand([0,0,1,0,1,1,0,1], base=2)

separator("Hexadecimal 2D — base 16 (D = 13)")
place_value_expand([2, 13], base=16)

All three results should be **45**. Three different ways to write the same quantity.

### Python's built-in conversions

Python has shorthand for the most common cases:

In [ ]:
v = 45

separator("Python built-ins: one value, three forms")
print(f"  decimal  : {v}")
print(f"  binary   : {bin(v)}        → stripped: {v:08b}")
print(f"  hex      : {hex(v)}         → uppercase: 0x{v:02X}")

separator("Parsing strings back to integers")
print(f"  int('101101',  2) = {int('101101', 2)}")
print(f"  int('2D',     16) = {int('2D', 16)}")
print(f"  int('45',     10) = {int('45', 10)}")

**Key format strings to remember:**

| Format | Meaning | Example (`v=45`) |
|--------|---------|------------------|
| `f"{v:b}"` | binary, no padding | `101101` |
| `f"{v:08b}"` | binary, 8 digits, zero-padded | `00101101` |
| `f"{v:x}"` | hex, lowercase | `2d` |
| `f"{v:02X}"` | hex, uppercase, 2 digits | `2D` |
| `f"{v:d}"` | decimal (default) | `45` |

---
## Section 3 — Conversion by Hand: Decimal ↔ Binary

Understanding *how* conversion works matters more than memorizing shortcuts.

In [ ]:
def decimal_to_binary_steps(n):
    """
    Convert n to binary using repeated subtraction of powers of 2.
    Shows every step so you can see the algorithm.
    """
    if n == 0:
        print("  0 → 0")
        return "0"

    # Find the highest power of 2 that fits
    bits = n.bit_length()  # number of binary digits needed
    result_bits = []
    remainder = n
    print(f"  Converting {n} to binary:")
    for pos in range(bits - 1, -1, -1):
        power_val = 2 ** pos
        if remainder >= power_val:
            result_bits.append(1)
            remainder -= power_val
            print(f"    position {pos}: 2^{pos}={power_val} fits  → bit=1, remainder={remainder}")
        else:
            result_bits.append(0)
            print(f"    position {pos}: 2^{pos}={power_val} too large → bit=0, remainder={remainder}")

    binary_str = ''.join(str(b) for b in result_bits)
    print(f"  Result: {binary_str}")
    return binary_str

separator("45 → binary")
result = decimal_to_binary_steps(45)
print(f"\n  Verify: int('{result}', 2) = {int(result, 2)}")

In [ ]:
# Alternative method: repeated division by 2
# Each remainder is one bit, read bottom-up.

def decimal_to_binary_division(n):
    """Convert via repeated division — the classic pencil-and-paper method."""
    if n == 0:
        return "0"
    remainders = []
    original = n
    print(f"  Dividing {n} repeatedly by 2:")
    while n > 0:
        q, r = divmod(n, 2)
        print(f"    {n} ÷ 2 = {q}  remainder {r}")
        remainders.append(r)
        n = q
    binary_str = ''.join(str(r) for r in reversed(remainders))
    print(f"  Remainders bottom-up: {binary_str}")
    return binary_str

separator("27 → binary (division method)")
decimal_to_binary_division(27)

### Exercise 3.1
Use one of the functions above to convert these values. Then verify with Python's `bin()`.

1. `decimal_to_binary_division(200)`
2. `decimal_to_binary_division(127)`
3. `decimal_to_binary_division(255)`

In [ ]:
# Your work here


---
## Section 4 — Nibbles and Bytes: the Hex↔Binary Bridge

One hex digit = one nibble = 4 bits.  
Two hex digits = one byte = 8 bits.

This is why hex is so useful: it's a compact, exact translation.

In [ ]:
# The 16 nibble patterns — memorise these and hex becomes transparent
HEX_DIGITS = '0123456789ABCDEF'

separator("All 16 nibble ↔ hex mappings")
print(f"  {'Decimal':>8}  {'Hex':>4}  {'Binary nibble':>14}")
print(f"  {'-------':>8}  {'---':>4}  {'-------------':>14}")
for i in range(16):
    print(f"  {i:>8}  {HEX_DIGITS[i]:>4}  {i:>14b}" if i == 0 else
          f"  {i:>8}  {HEX_DIGITS[i]:>4}  {i:>14b}")

# Fix the formatting to zero-pad nibbles
separator("Corrected with zero-padded nibbles")
print(f"  {'Decimal':>8}  {'Hex':>4}  {'Nibble':>8}")
for i in range(16):
    print(f"  {i:>8}  {HEX_DIGITS[i]:>4}  {i:>08b}"[-15:])

# Actually let's keep it clean
separator("Clean version")
print(f"  {'Dec':>4}  {'Hex':>4}  {'4-bit nibble':>12}")
for i in range(16):
    print(f"  {i:>4}  {HEX_DIGITS[i]:>4}  {i:>12b}" if False else
          f"  {i:4d}  {HEX_DIGITS[i]:>4}  {i:04b}")

In [ ]:
def byte_to_hex(byte_val):
    """
    Convert a byte (0–255) to its two-digit hex representation
    by splitting into two nibbles.
    """
    v = byte_val & 0xFF

    # Extract the two nibbles using bit operations
    high_nibble = (v >> 4) & 0xF   # upper 4 bits
    low_nibble  =  v       & 0xF   # lower 4 bits

    high_bits = f"{high_nibble:04b}"  # 4-bit binary string
    low_bits  = f"{low_nibble:04b}"

    high_hex  = HEX_DIGITS[high_nibble]
    low_hex   = HEX_DIGITS[low_nibble]

    print(f"  Byte value : {v} (decimal)")
    print(f"  Binary     : {v:08b}")
    print(f"  Split      : [{high_bits}] [{low_bits}]  (high nibble | low nibble)")
    print(f"  Nibble dec : {high_nibble} and {low_nibble}")
    print(f"  Hex digits : {high_hex} and {low_hex}")
    print(f"  Result     : 0x{high_hex}{low_hex}")
    return f"0x{high_hex}{low_hex}"

separator("00101101 = 45 → hex")
byte_to_hex(45)

separator("01010111 = 87 → hex  (appears in AES examples as 0x57)")
byte_to_hex(0b01010111)

In [ ]:
def hex_to_byte(hex_str):
    """
    Convert a two-character hex string to its 8-bit binary representation.
    Shows both nibbles explicitly.
    """
    hex_str = hex_str.upper().lstrip('0X')
    assert len(hex_str) <= 2, "Expecting at most two hex digits (one byte)"
    hex_str = hex_str.zfill(2)  # ensure two digits

    high_char = hex_str[0]
    low_char  = hex_str[1]

    high_val = int(high_char, 16)
    low_val  = int(low_char,  16)
    byte_val = (high_val << 4) | low_val

    print(f"  Hex input  : 0x{hex_str}")
    print(f"  High digit : {high_char} = {high_val} = {high_val:04b}")
    print(f"  Low digit  : {low_char} = {low_val} = {low_val:04b}")
    print(f"  Combined   : {high_val:04b}{low_val:04b} = {byte_val}")
    return byte_val

separator("0xAE → binary")
hex_to_byte("AE")

separator("0xC1 → binary")
hex_to_byte("C1")

### Exercise 4.1 — Spot the byte

For each hex value below, call `hex_to_byte()` and then manually verify the binary by reading the nibbles:

1. `0xFF` — the maximum byte value
2. `0x0F` — only the low nibble set
3. `0xF0` — only the high nibble set
4. `0x57` — the AES example from the tutorial

In [ ]:
# Your work here


---
## Section 5 — Fixed-Width Arithmetic and Overflow

A byte has exactly 8 bits. Values 0–255. When you add 1 to 255, the result (256) needs 9 bits — but the container only holds 8. The 9th bit is *dropped* and the value wraps to 0.

The Python trick is the **mask**: `& 0xFF` after every operation.

In [ ]:
def byte_add(a, b):
    """
    Add two values as if in an 8-bit unsigned register.
    Shows overflow when it occurs.
    """
    a = a & 0xFF
    b = b & 0xFF
    true_sum = a + b
    byte_sum = true_sum & 0xFF  # keep only the lowest 8 bits
    overflowed = true_sum > 255

    print(f"  {a:3d} (0x{a:02X}, {a:08b})")
    print(f"+ {b:3d} (0x{b:02X}, {b:08b})")
    print(f"  {'─'*34}")
    if overflowed:
        carry_bit = true_sum >> 8  # the bit that overflowed
        print(f"  True sum = {true_sum} = {true_sum:09b}  (9 bits — bit 8 is the carry)")
        print(f"  Carry bit {carry_bit} is DROPPED (no position 8 in a byte!)")
    print(f"  Result = {byte_sum:3d} (0x{byte_sum:02X}, {byte_sum:08b})", end="")
    if overflowed:
        print("  ← OVERFLOW: wrapped around")
    else:
        print()
    return byte_sum

separator("Normal addition: 100 + 45")
byte_add(100, 45)

separator("Overflow: 200 + 100")
byte_add(200, 100)

separator("Maximum overflow: 255 + 1")
byte_add(255, 1)

In [ ]:
# Simulating the overflow counter from the tutorial
# Count from 253 to 2 and watch the wraparound

separator("Counting near the overflow boundary")
val = 253
for _ in range(6):
    note = "  ← OVERFLOW" if val == 0 else ""
    print(f"  {val:>3}   0x{val:02X}   {val:08b}{note}")
    val = (val + 1) & 0xFF

In [ ]:
# Why does & 0xFF work?
# 0xFF = 11111111 in binary — a mask that keeps only the lowest 8 bits.

separator("How the mask works")
true_sum = 256  # 100000000 — 9 bits
mask     = 0xFF # 011111111 — 8 ones

print(f"  true_sum = {true_sum} = {true_sum:09b}")
print(f"  mask     = {mask}  = {mask:09b}   (0xFF)")
print(f"  AND      = {true_sum & mask}    = {(true_sum & mask):09b}   (bit 8 gone)")

print()
print("  The AND with 0xFF zeros out any bit above position 7.")
print("  This is equivalent to keeping the remainder after dividing by 256.")
print(f"  256 mod 256 = {256 % 256}")
print(f"  300 mod 256 = {300 % 256}  → same as {300 & 0xFF}")

### Exercise 5.1 — Predict the overflow

Before running, *predict* what the byte result will be:
1. `byte_add(255, 255)`  — what is 510 mod 256?
2. `byte_add(128, 128)`  — what is 256 mod 256?
3. `byte_add(200, 56)`   — does this overflow?

In [ ]:
# Predict first, then run


---
## Section 6 — Conversion Challenge: Build Your Own Converter

Now put it together. Write a function that converts any integer to any base (2–16) and shows the work.

In [ ]:
def to_base(n, base, width=0):
    """
    Convert integer n to the given base (2–16).
    Shows the repeated-division working.
    width: minimum digits (zero-padded).
    """
    assert 2 <= base <= 16, "Base must be 2–16"
    DIGITS = '0123456789ABCDEF'

    if n == 0:
        result = '0'
    else:
        result_chars = []
        remainder = n
        print(f"  Converting {n} to base {base}:")
        while remainder > 0:
            q, r = divmod(remainder, base)
            print(f"    {remainder} ÷ {base} = {q}  remainder {r}  → digit '{DIGITS[r]}'")
            result_chars.append(DIGITS[r])
            remainder = q
        result = ''.join(reversed(result_chars))

    if width:
        result = result.zfill(width)
    print(f"  Result (base {base}): {result}")
    return result

separator("45 → base 2")
to_base(45, base=2, width=8)

separator("45 → base 16")
to_base(45, base=16, width=2)

In [ ]:
# Verify a full round-trip: decimal → binary → decimal
original = 173
binary_str = to_base(original, base=2, width=8)
print()
back = int(binary_str, 2)
print(f"  Round-trip: {original} → '{binary_str}' → {back}")
assert original == back, "Round-trip failed!"
print("  Round-trip verified.")

### Exercise 6.1

Use `to_base()` to solve the worked examples from the tutorial. Verify each answer:

1. Convert `101101` binary → decimal (you already know it's 45)
2. Convert 45 → binary (should be `00101101`)
3. Convert `01010111` binary → hex (the `0x57` AES value)
4. Convert `0xC1` → binary

In [ ]:
# Tip: int('101101', 2) to go binary→decimal first, then to_base(45, 16)


---
## Section 7 — Real Cryptography: Hex in the Wild

Cryptographic output is almost always displayed in hexadecimal. A 128-bit AES key is 16 bytes = 32 hex characters. A SHA-256 hash is 32 bytes = 64 hex characters.

Let's decode some real cryptographic byte sequences.

In [ ]:
def decode_hex_sequence(hex_string, label=""):
    """
    Parse a hex string (like a key or hash output) into its individual bytes.
    Shows decimal, hex, and binary for each byte.
    """
    # Strip spaces and 0x prefix
    clean = hex_string.replace(' ', '').replace('0x', '').upper()
    assert len(clean) % 2 == 0, "Hex string must have an even number of characters"

    bytes_list = [int(clean[i:i+2], 16) for i in range(0, len(clean), 2)]

    if label:
        print(f"  {label}")
    print(f"  {len(bytes_list)} bytes ({len(bytes_list)*8} bits)")
    print(f"  {'Byte':>5}  {'Dec':>4}  {'Hex':>4}  {'Binary':>10}  {'ASCII':>5}")
    print(f"  {'-----':>5}  {'---':>4}  {'---':>4}  {'--------':>10}  {'-----':>5}")
    for i, b in enumerate(bytes_list):
        ascii_char = chr(b) if 32 <= b < 127 else '.'
        print(f"  [{i:>3}]  {b:>4}  0x{b:02X}  {b:>10b}  {ascii_char:>5}")
    return bytes_list

separator("An AES-128 key (16 bytes = 128 bits)")
decode_hex_sequence("2b7e151628aed2a6abf7158809cf4f3c",
                    label="AES test vector key (FIPS 197 Appendix B)")

In [ ]:
separator("A SHA-256 hash of the empty string (32 bytes = 256 bits)")
sha256_empty = "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
decode_hex_sequence(sha256_empty, label='SHA-256("") — hash of the empty string')

In [ ]:
# Python's hashlib lets you compute real hashes
import hashlib

separator("Computing SHA-256 with Python")
message = b"Hello, cryptography!"
digest = hashlib.sha256(message).digest()   # bytes object
hex_digest = digest.hex()                   # hex string

print(f"  Message : {message}")
print(f"  SHA-256 : {hex_digest}")
print(f"  Length  : {len(digest)} bytes = {len(digest)*8} bits = {len(hex_digest)} hex chars")

print()
print("  First 4 bytes broken down:")
for b in digest[:4]:
    show_byte(b)

In [ ]:
# Notice that even one character change completely transforms the hash — the avalanche effect
separator("Avalanche effect: tiny input change → completely different hash")

msg1 = b"Hello, cryptography!"
msg2 = b"Hello, cryptography."   # only the ! changed to .

h1 = hashlib.sha256(msg1).hexdigest()
h2 = hashlib.sha256(msg2).hexdigest()

print(f"  msg1: {msg1}")
print(f"  H1  : {h1}")
print()
print(f"  msg2: {msg2}")
print(f"  H2  : {h2}")

# Count differing bits
b1 = int(h1, 16)
b2 = int(h2, 16)
xor_result = b1 ^ b2
differing_bits = bin(xor_result).count('1')
print(f"\n  Bits that differ: {differing_bits} out of 256 ({differing_bits/256*100:.1f}%)")
print("  (A good hash changes ~50% of bits even for a 1-bit input change.")

---
## Section 8 — Byte Representation in Memory

Python's `bytes` type is how binary data actually travels through programs. Understanding it reinforces everything above.

In [ ]:
separator("bytes objects in Python")

# Three ways to create the same bytes object
b_from_list    = bytes([72, 101, 108, 108, 111])   # from decimal list
b_from_hex     = bytes.fromhex("48656c6c6f")        # from hex string
b_from_literal = b"Hello"                           # byte literal

print(f"  From decimal list : {b_from_list}")
print(f"  From hex string   : {b_from_hex}")
print(f"  From literal      : {b_from_literal}")
print(f"  All equal?        : {b_from_list == b_from_hex == b_from_literal}")

print()
print("  Iterating over bytes gives integers:")
for i, byte_val in enumerate(b_from_literal):
    print(f"    index {i}: {byte_val:3d} = 0x{byte_val:02X} = {byte_val:08b} = '{chr(byte_val)}'")

In [ ]:
def bytes_to_hex_table(data: bytes, label=""):
    """
    Pretty-print bytes the way cryptographic tools typically show them:
    offset | hex pairs | ASCII
    """
    width = 16  # bytes per row
    if label:
        print(f"  {label}")
    print(f"  {'Offset':>6}  {'Hex':^47}  {'ASCII':^16}")
    print(f"  {'------':>6}  {'---':^47}  {'-----':^16}")
    for row_start in range(0, len(data), width):
        chunk = data[row_start:row_start + width]
        hex_part   = ' '.join(f"{b:02X}" for b in chunk)
        ascii_part = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f"  {row_start:06X}   {hex_part:<47}  {ascii_part}")

separator("Hex dump of the AES test vector key")
key_bytes = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
bytes_to_hex_table(key_bytes, label="AES-128 key (FIPS 197 Appendix B)")

---
## Section 9 — Summary and Final Challenge

### What you have built

| Concept | Python technique |
|---------|------------------|
| Place-value expansion | `place_value_expand(digits, base)` |
| Decimal → binary | `to_base(n, 2)` / `bin(n)` / `f"{n:08b}"` |
| Binary → decimal | `int(s, 2)` |
| Decimal → hex | `to_base(n, 16)` / `hex(n)` / `f"{n:02X}"` |
| Hex → decimal | `int(s, 16)` |
| Nibble splitting | `(v >> 4) & 0xF` and `v & 0xF` |
| Fixed-width wrap | `(v + k) & 0xFF` |
| Bytes in Python | `bytes`, `bytes.fromhex()`, `.hex()` |

### Final Challenge — Decode the Hidden Message

Below is a sequence of byte values stored as a hex string. Decode it to reveal a message.

**Rules:**
1. Split the hex string into bytes
2. Each byte in the range 32–126 is a printable ASCII character
3. Read the ASCII values to find the message

In [ ]:
# The hidden message — decode it
mystery_hex = "4269747320616e64206279746573206172652074686520616c706861626574206f662063727970746f677261706879"

# Step 1: convert hex string to bytes
mystery_bytes = bytes.fromhex(mystery_hex)

# Step 2: decode as UTF-8 / ASCII
print("Decoded message:")
print()
print(" " * 2 + mystery_bytes.decode('ascii'))
print()
print(f"That was {len(mystery_bytes)} bytes = {len(mystery_bytes)*8} bits = {len(mystery_hex)} hex characters.")

In [ ]:
# Challenge extension: encode your own message as hex
your_message = "Cryptography starts with bits."

encoded = your_message.encode('ascii').hex()
print(f"Your message  : {your_message}")
print(f"As hex        : {encoded}")
print(f"Length        : {len(your_message)} chars → {len(your_message)} bytes → {len(encoded)} hex chars")

# Verify round-trip
decoded_back = bytes.fromhex(encoded).decode('ascii')
assert decoded_back == your_message
print("Round-trip verified.")

---
## What Comes Next

Now that bits and bytes are visible, **Module 02 — Bitwise Operations as Algebra** introduces the operations that transform them: XOR (⊕), AND (&), OR (|), NOT (~), shifts, and rotations.

XOR in particular shows up in *every* layer of modern cryptography — stream ciphers, block cipher round functions, hash mixing, and authenticated encryption.

The Python companion for Module 02 (`notebook_02_bitwise_operations_as_algebra.ipynb`) picks up right where this one ends.

---
*Mathematics of Cryptography — Python Companion Series*